# Example 6.1

The reference surface is the ellipsoid $p_x^2/4+p_y^2/4+p_z^2/9=1$.
The embedding $r(t,p)=((1-t/2)p_x,(1-t/2)p_y,(1-2t/3)p_z)$ reaches the unit sphere at $t=1$.

Based on `IsoEmb_Ref/IsoEmbRate_paper.ipynb`: nodal Lagrange startup values,
the background $g_{M_h}=R_h(a^*g_M)$ from the closest-point Jacobian,
and BDF3 with six rigid-motion constraints. As in the actual source code,
the RHS metric rate uses **FE endpoint derivatives**; this differs from the
analytic metric-rate interpolation described in the paper's numerical experiments.

Run all cells in order in a Python kernel with NGSolve and NumPy.
All required functions are included. Restart and run all after changing parameters.


## 1. Parameters, mesh and finite-element spaces


In [ ]:
import numpy as np
from ngsolve import *
from netgen.occ import Pnt, Axis, Face, Wire, SplineApproximation, OCCGeometry, Z
from netgen.meshing import MeshingStep

# Defaults from IsoEmbRate_paper.ipynb. Use r=k=5 or 6 for high-order runs.
h = 0.5                 # Maximum mesh size
r = 1                   # Regge metric degree
k = 1                   # H1 embedding degree
tau = 0.05
T = 1.0
num_threads_asm = 4
show_surface = False    # Optional ngsolve.webgui display in the last cell
inverse = "pardiso"      # Use "umfpack" if provided instead by your NGSolve build
Ax, Ay, Az = 2, 2, 3      # Fixed reference surface for this example
metric_bonus_intorder = 10
quad_order_asm = 30

if not (np.isfinite(tau) and np.isfinite(T) and tau > 0 and T > 0):
    raise ValueError("tau and T must be finite and positive")
N_steps = int(round(T / tau))
if N_steps < 3 or not np.isclose(N_steps*tau, T, rtol=1e-12, atol=0):
    raise ValueError("BDF3 requires integer T/tau >= 3")
tau = T / N_steps
SetNumThreads(num_threads_asm)

# Retain the original spline-of-revolution surface mesh.
points_on_curve = [Pnt(0, Ay*np.sin(phi), Az*np.cos(phi))
                   for phi in np.linspace(0, np.pi, 200)]
face = Face(Wire([SplineApproximation(points_on_curve, tol=1e-8)]))
surface = face.Revolve(Axis((0, 0, 0), Z), 360)
mesh = Mesh(OCCGeometry(surface).GenerateMesh(
    maxh=h, perfstepsend=MeshingStep.MESHSURFACE, grading=0.5))
surface_region = mesh.Boundaries('.*')
ds_asm = ds(intrules={ET.TRIG: IntegrationRule(ET.TRIG, order=quad_order_asm)})
fesG = HCurlCurl(mesh, order=r)
fesC = NumberSpace(mesh)
fesV = VectorH1(mesh, order=k)
fes = H1(mesh, order=k)
fesMix = fesV*fesC*fesC*fesC*fesC*fesC*fesC
print(f"Embedding DOFs: {fesV.ndof}; time steps: {N_steps}")

quad_order_ref = 20       # Verified below against the order-40 triangle rule
cp_newton_iterations = 5


## 2. Reference geometry and nodal initialization

The safeguarded closest-point solver supplies independent reference samples. Native coefficient functions supply geometry at arbitrary Regge quadrature points. The two evaluations are checked against each other.


In [ ]:
from time import perf_counter

def project_ellipsoid(query_points, axes, tol=1e-15, max_iter=50):
    """Return (phi, psi, points, info), with the original projection info keys.

    Solve sum(a_i^2 q_i^2 / (a_i^2 + lambda)^2) = 1 by safeguarded
    Newton iteration. The branch lambda >= -min(a_i^2) gives a global
    closest point. This is a normal projection, not radial rescaling.
    tol bounds the estimated remaining Cartesian correction (length units);
    convergence also requires the dimensionless surface residual <= 8 eps.
    At interior points with multiple closest points, choose one deterministically.
    """
    start = perf_counter()
    query = np.asarray(query_points, dtype=float)
    axes = np.asarray(axes, dtype=float)
    if query.ndim != 2 or query.shape[1] != 3 or not np.isfinite(query).all():
        raise ValueError('query_points must be a finite (N, 3) array')
    if axes.shape != (3,) or not np.isfinite(axes).all() or np.any(axes <= 0):
        raise ValueError('axes must contain three finite positive semiaxes')
    if not np.isfinite(tol) or tol <= 0 or not isinstance(max_iter, int) or max_iter < 1:
        raise ValueError('tol must be positive and max_iter must be a positive integer')

    scale = float(np.max(axes))
    a = axes / scale
    a2 = a*a
    q = query / scale
    amin2 = float(np.min(a2))
    d = a2 - amin2
    shortest = d == 0
    count = len(q)
    p = np.zeros_like(q)
    s = np.full(count, amin2)
    steps = np.full(count, np.inf)
    converged = np.zeros(count, dtype=bool)
    residual_tol = 8 * np.finfo(float).eps

    on_axis = np.all(q[:, shortest] == 0, axis=1)
    axis_ids = np.flatnonzero(on_axis)
    pole = np.zeros((len(axis_ids), 3))
    pole[:, ~shortest] = a2[~shortest] * q[axis_ids][:, ~shortest] / d[~shortest]
    remaining = 1 - np.sum((pole/a)**2, axis=1)
    special = axis_ids[remaining >= 0]
    p[special] = pole[remaining >= 0]
    first_short = np.flatnonzero(shortest)[0]
    p[special, first_short] = a[first_short] * np.sqrt(remaining[remaining >= 0])
    s[special] = 0
    steps[special] = 0
    converged[special] = True

    low = np.zeros(count)
    high = np.maximum(amin2, np.linalg.norm(a*q, axis=1))
    iterations = 0
    for iteration in range(max_iter):
        ids = np.flatnonzero(~converged)
        if not len(ids):
            break
        iterations = iteration + 1
        den = d + s[ids, None]
        p[ids] = a2 * q[ids] / den
        terms = (p[ids]/a)**2
        f = np.sum(terms, axis=1) - 1
        df = -2 * np.sum(terms/den, axis=1)
        delta = f / df
        steps[ids] = scale * np.abs(delta) * np.linalg.norm(p[ids]/den, axis=1)
        done = (np.abs(f) <= residual_tol) & (steps[ids] <= tol)
        converged[ids[done]] = True
        if iteration == max_iter - 1:
            break
        low[ids] = np.where(f > 0, s[ids], low[ids])
        high[ids] = np.where(f < 0, s[ids], high[ids])
        trial = s[ids] - delta
        safe = np.isfinite(trial) & (trial > low[ids]) & (trial < high[ids])
        trial = np.where(safe, trial, (low[ids] + high[ids])/2)
        s[ids[~done]] = trial[~done]

    surface_residuals = np.abs(np.sum((p/a)**2, axis=1) - 1)
    normal_residuals = scale * np.linalg.norm(q - p - (s - amin2)[:, None]*p/a2, axis=1)
    converged &= (np.isfinite(p).all(axis=1)
                  & (surface_residuals <= residual_tol)
                  & (normal_residuals <= 8*np.finfo(float).eps
                     * np.maximum(scale, np.linalg.norm(query, axis=1))))
    failed = np.flatnonzero(~converged)
    points = scale*p
    phi_values = np.arctan2(np.hypot(p[:, 0]/a[0], p[:, 1]/a[1]), p[:, 2]/a[2])
    psi_values = np.arctan2(p[:, 1]/a[1], p[:, 0]/a[0]) % (2*np.pi)
    info = {
        'non_converged_indices': failed,
        'final_delta_norms': steps[failed],
        'iterations': iterations,
        'converged_count': int(np.count_nonzero(converged)),
        'max_surface_residual': float(np.max(surface_residuals, initial=0)),
        'max_normal_residual': float(np.max(normal_residuals, initial=0)),
        'max_step_norm': float(np.max(steps, initial=0)),
        'elapsed_seconds': perf_counter() - start,
    }
    return phi_values, psi_values, points, info

def interpolate_surface_lagrange(fes, evaluate, order):
    """Interpolate point values in the oriented triangular H1 basis."""
    if not isinstance(order, (int, np.integer)) or order < 1:
        raise ValueError('order must be a positive integer')
    mesh = fes.mesh
    if (mesh.dim != 3 or mesh.ne != 0 or mesh.GetCurveOrder() != 1
            or mesh.deformation is not None):
        raise ValueError('expected an undeformed, piecewise-flat surface mesh in R^3')
    if fes.type != 'h1ho' or fes.dim != 1:
        raise ValueError('fes must be a scalar H1 space')

    nodes = [(i / order, j / order)
             for i in range(order + 1) for j in range(order + 1 - i)]

    ir = IntegrationRule(nodes, [1.0] * len(nodes))
    coords = CF((x, y, z))
    elements = []
    physical_points = []
    matrices = {}
    for element in mesh.Elements(BND):
        if element.type != ET.TRIG:
            raise ValueError('only triangular surface elements are supported')
        fe = fes.GetFE(element)
        dofs = np.asarray(fes.GetDofNrs(element), dtype=int)
        if fe.order != order or fe.ndof != len(nodes) or np.any(dofs < 0):
            raise ValueError('expected a uniform, unconstrained local P_k space')
        matrix = np.array([fe.CalcShape(u, v).NumPy().copy() for u, v in nodes])

        key = matrix.tobytes()
        if key not in matrices:
            matrices[key] = matrix
        elements.append((dofs, key))
        physical_points.append(np.asarray(coords(mesh.GetTrafo(element)(ir))))
    if not elements:
        raise ValueError('the surface mesh is empty')

    values = np.asarray(evaluate(np.concatenate(physical_points)), dtype=float)
    if values.ndim == 1:
        values = values[:, None]
    npoints = len(elements) * len(nodes)
    if (values.ndim != 2 or values.shape[0] != npoints or values.shape[1] == 0
            or not np.isfinite(values).all()):
        raise ValueError('evaluate must return finite (N,) or (N, d) values')
    values = values.reshape(len(elements), len(nodes), -1)
    coefficients = np.zeros((fes.ndof, values.shape[2]))
    counts = np.zeros(fes.ndof, dtype=int)
    for (dofs, key), local_values in zip(elements, values):
        local_coefficients = np.linalg.solve(matrices[key], local_values)
        np.add.at(coefficients, dofs, local_coefficients)
        np.add.at(counts, dofs, 1)
    if np.any(counts == 0):
        raise ValueError('fes contains DOFs outside the triangular surface')
    coefficients /= counts[:, None]

    max_error = max(float(np.max(np.abs(matrices[key] @ coefficients[dofs] - local_values)))
                    for (dofs, key), local_values in zip(elements, values))
    scale = max(1.0, float(np.max(np.abs(values))))
    if not np.isfinite(coefficients).all() or max_error > 1e-11 * scale:
        raise RuntimeError('nodal interpolation failed: inconsistent traces or ill-conditioned basis')
    return coefficients, {
        'sample_points': npoints,
        'max_nodal_error': max_error,
    }

import ngsolve as ng

def ellipsoid_cp_jacobian_cf(axes, iterations=5):
    """Return the closest-point map and its analytical Jacobian as native CFs.

    Newton solves sum(a_i^2*x_i^2/(a_i^2+lambda)^2)=1 starting at lambda=0.
    This implementation is for the tubular neighborhood of the ellipsoid,
    as in this example's meshes. Validate it with check_geometry_cf below.
    'Exact' geometry here means no finite-element interpolation of the map;
    Newton iteration, quadrature and floating-point tolerances still apply.
    """
    axes = np.asarray(axes, dtype=float)
    if axes.shape != (3,) or not np.isfinite(axes).all() or np.any(axes <= 0):
        raise ValueError('axes must contain three finite positive semiaxes')
    if not isinstance(iterations, int) or iterations < 1:
        raise ValueError('iterations must be a positive integer')
    axes2 = tuple(float(a*a) for a in axes)
    coord = (ng.x, ng.y, ng.z)
    lam = ng.CF(0.0)
    for _ in range(iterations):
        den = tuple(a2 + lam for a2 in axes2)
        value = sum(axes2[i]*coord[i]**2 / den[i]**2 for i in range(3)) - 1
        derivative = -2*sum(axes2[i]*coord[i]**2 / den[i]**3 for i in range(3))
        lam = (lam - value/derivative).Compile(maxderiv=0)

    diagonal = tuple(axes2[i] / (axes2[i]+lam) for i in range(3))
    point = ng.CF(tuple(diagonal[i]*coord[i] for i in range(3)))
    normal = ng.CF(tuple(point[i]/axes2[i] for i in range(3)))
    bn = ng.CF(tuple(diagonal[i]*normal[i] for i in range(3)))
    B = ng.CF(tuple(diagonal[i] if i == j else 0
                    for i in range(3) for j in range(3)), dims=(3, 3))
    jacobian = B - ng.OuterProduct(bn, bn)/ng.InnerProduct(normal, bn)
    return point.Compile(maxderiv=0), jacobian.Compile(maxderiv=0)

def check_geometry_cf(point_cf, jacobian_cf, fesir, coords, points, axes,
                      tolerance=5e-13):
    """Compare native CFs against the independent checked NumPy projection.

    `coords` and `points` are the existing Cell 2 quadrature arrays. Sampling
    on this same IntegrationRuleSpace preserves its ordering. No sampled
    values from this check are fed into Regge interpolation.
    """
    axes2 = np.asarray(axes, dtype=float)**2
    normal = points / axes2
    lam = np.sum((coords-points)*normal, axis=1) / np.sum(normal**2, axis=1)
    diagonal = 1.0 / (1.0 + lam[:, None]/axes2)
    bn = diagonal * normal
    jacobian = (diagonal[:, :, None]*np.eye(3)
                - bn[:, :, None]*bn[:, None, :]
                / np.sum(normal*bn, axis=1)[:, None, None])
    sample = ng.GridFunction(fesir)
    region = fesir.mesh.Boundaries('.*')
    errors = dict(point=0.0, jacobian=0.0)
    for name, expressions, expected in (
        ('point', [point_cf[i] for i in range(3)], points),
        ('jacobian', [jacobian_cf[i, j] for i in range(3) for j in range(3)],
         jacobian.reshape(-1, 9)),
    ):
        for index, expression in enumerate(expressions):
            sample.Interpolate(expression, region)
            values = sample.vec.FV().NumPy()
            if not np.isfinite(values).all():
                raise RuntimeError('non-finite closest-point coefficient function')
            errors[name] = max(errors[name], float(np.max(np.abs(values-expected[:, index]))))
    if max(errors.values()) > tolerance:
        raise RuntimeError('closest-point CF check failed: {}; check Newton convergence'.format(errors))
    return errors


In [ ]:
from ngsolve.comp import IntegrationRuleSpaceSurface

fesir = IntegrationRuleSpaceSurface(mesh, order=quad_order_ref, definedon=surface_region)
irs = fesir.GetIntegrationRules()
if str(irs[ET.TRIG]) != str(IntegrationRule(ET.TRIG, order=40)):
    raise RuntimeError("Expected the order-40 triangle quadrature rule")
coords_ir = GridFunction(fesir**3)
coords_ir.Interpolate(CF((x, y, z)), surface_region)
coords_3d = np.column_stack([c.vec.FV().NumPy() for c in coords_ir.components])
_, _, points, projection_info = project_ellipsoid(coords_3d, (Ax, Ay, Az))
if projection_info['non_converged_indices'].size:
    raise RuntimeError("Closest-point projection did not converge")
X_ref0 = GridFunction(fesir**3)
X_ref1 = GridFunction(fesir**3)
for i, factor in enumerate((1/2, 1/2, 1/3)):
    X_ref0.components[i].vec.FV().NumPy()[:] = points[:, i]
    X_ref1.components[i].vec.FV().NumPy()[:] = factor*points[:, i]

# Native coefficient functions can also be evaluated at Regge edge points.
a_geometry, Da_geometry = ellipsoid_cp_jacobian_cf(
    (Ax, Ay, Az), iterations=cp_newton_iterations)
geometry_check = check_geometry_cf(
    a_geometry, Da_geometry, fesir, coords_3d, points, (Ax, Ay, Az))
print("Closest-point/Jacobian check:", geometry_check)

mesh.UnsetDeformation()
def exact_initial_values(node_coords):
    _, _, node_points, node_info = project_ellipsoid(
        node_coords, (Ax, Ay, Az), tol=1e-15, max_iter=50,
    )
    if node_info['non_converged_indices'].size:
        raise RuntimeError("Closest-point projection failed at a Lagrange node")
    return node_points

initial_coeffs, initial_interp_info = interpolate_surface_lagrange(
    fes, exact_initial_values, k,
)
Xh0 = GridFunction(fesV)
Xh1 = GridFunction(fesV)
for i, factor in enumerate((1/2, 1/2, 1/3)):
    Xh0.components[i].vec.FV().NumPy()[:] = initial_coeffs[:, i]

    Xh1.components[i].vec.data = factor * Xh0.components[i].vec
Exa_ptr = GridFunction(fesV)
Exa_ptr.vec.data = Xh1.vec - Xh0.vec
print(f"Lagrange initial data: max nodal error={initial_interp_info['max_nodal_error']:.3e}")

nsurf = specialcf.normal(3)
NN = OuterProduct(nsurf, nsurf)
Ptau = Id(3) - NN
J0_geometry = Da_geometry * Ptau
J = J0_geometry
G = J.trans * J
gMh = GridFunction(fesG)
gMh.Set(
    G, definedon=mesh.Boundaries('.*'),
    dual=True,
    bonus_intorder=metric_bonus_intorder,
)
g_inv = Ptau * Inv(gMh + NN) * Ptau
sq_det = sqrt(Det(gMh + NN))

Pos0 = GridFunction(fesV)
Pos1 = GridFunction(fesV)
Pos_1 = GridFunction(fesV)
Pos_2 = GridFunction(fesV)
Pos_hat = GridFunction(fesV)

Solution= GridFunction(fesMix)

told = 0.0
Pos_2.vec.data = (1 - told) * Xh0.vec + told * Xh1.vec
told = tau
Pos_1.vec.data = (1 - told) * Xh0.vec + told * Xh1.vec
told = 2 * tau
step_n = 2
Pos0.vec.data = (1 - told) * Xh0.vec + told * Xh1.vec


## 3. BDF3 time stepping

Startup values are interpolated at $0,\tau,2\tau$. The extrapolated embedding is used in both the strain and the six rigid-motion constraints. One linear system is solved per step.


In [ ]:
SetNumThreads(num_threads_asm)

Exa_ptg = GridFunction(fesG)
v, alp1, alp2, alp3, beta1, beta2, beta3 = fesMix.TrialFunction()
w, alp1_t, alp2_t, alp3_t, beta1_t, beta2_t, beta3_t = fesMix.TestFunction()

Jhat_cached = CacheCF(grad(Pos_hat).Trace())
g_inv_cached = CacheCF(g_inv)
mu_cached = CacheCF(sq_det)

dv = grad(v).Trace()
dw = grad(w).Trace()
strain_v = 0.5 * (Jhat_cached.trans * dv + dv.trans * Jhat_cached)
strain_w = 0.5 * (Jhat_cached.trans * dw + dw.trans * Jhat_cached)

lhs = BilinearForm(fesMix)
lhs += (
    2 * Trace(g_inv_cached * strain_v * g_inv_cached * strain_w)
    * mu_cached * ds_asm
)

lhs += (
    InnerProduct(w, Cross((alp1, alp2, alp3), Pos_hat)
                 + (beta1, beta2, beta3)) * sq_det
).Compile() * ds_asm
lhs += (
    InnerProduct(v, Cross((alp1_t, alp2_t, alp3_t), Pos_hat)
                 + (beta1_t, beta2_t, beta3_t)) * sq_det
).Compile() * ds_asm

Jhat_rhs = grad(Pos_hat).Trace()
strain_w_rhs = 0.5 * (Jhat_rhs.trans * dw + dw.trans * Jhat_rhs)
rhs = LinearForm(fesMix)
rhs += (
    Trace(g_inv * Exa_ptg * g_inv * strain_w_rhs) * sq_det
).Compile() * ds_asm

J0_rhs = grad(Xh0).Trace()
Jdelta_rhs = grad(Xh1).Trace() - J0_rhs
Jvelocity_rhs = grad(Exa_ptr).Trace()
F0_ptg = J0_rhs.trans * Jvelocity_rhs + Jvelocity_rhs.trans * J0_rhs
F1_ptg = Jdelta_rhs.trans * Jvelocity_rhs + Jvelocity_rhs.trans * Jdelta_rhs
ptg_const = GridFunction(fesG)
ptg_slope = GridFunction(fesG)

with TaskManager():
    ptg_const.Set(F0_ptg, definedon=surface_region, dual=True,
                  bonus_intorder=metric_bonus_intorder)
    ptg_slope.Set(F1_ptg, definedon=surface_region, dual=True,
                  bonus_intorder=metric_bonus_intorder)

    # Cache both fixed norms at the original order-40 quadrature points.
    normal = GridFunction(fesir**3)
    normal.Interpolate(specialcf.normal(3), definedon=surface_region)
    nh_values = np.column_stack([c.vec.FV().NumPy() for c in normal.components])
    nn_values = nh_values[:, :, None]*nh_values[:, None, :]
    ph_values = np.eye(3)-nn_values
    axes2 = np.array([Ax, Ay, Az], dtype=float)**2
    cp_normal = points/axes2
    lam = np.sum((coords_3d-points)*cp_normal, axis=1)/np.sum(cp_normal**2, axis=1)
    diagonal = 1/(1+lam[:, None]/axes2)
    bn = diagonal*cp_normal
    da_values = (diagonal[:, :, None]*np.eye(3)
                 - bn[:, :, None]*bn[:, None, :]
                 / np.sum(cp_normal*bn, axis=1)[:, None, None])
    j0_values = da_values @ ph_values
    g0_augmented = j0_values.transpose(0, 2, 1) @ j0_values + nn_values
    h_exact_values = ph_values @ np.linalg.inv(g0_augmented) @ ph_values
    mu_exact_values = np.sqrt(np.linalg.det(g0_augmented))
    tensor_sample = GridFunction(fesir**9)
    density_sample = GridFunction(fesir)
    diagnostic_ds = ds(intrules=irs)

    def sample_tensor(tensor):
        tensor_sample.Interpolate(tensor.Reshape((9,)), definedon=surface_region)
        return np.column_stack([c.vec.FV().NumPy() for c in tensor_sample.components]).reshape(-1, 3, 3)

    h_discrete_values = sample_tensor(g_inv)
    density_sample.Interpolate(sq_det, definedon=surface_region)
    mu_discrete_values = density_sample.vec.FV().NumPy().copy()
    initial_discrete_metric = GridFunction(fesG)
    initial_discrete_metric.Set(J0_rhs.trans*J0_rhs, definedon=surface_region,
                               dual=True, bonus_intorder=metric_bonus_intorder)
    target_metric = GridFunction(fesG)
    metric_history = []

    def metric_norm(defect, inverse_metric, weight):
        hd = inverse_metric @ defect
        density_sample.vec.FV().NumPy()[:] = np.einsum('nij,nji->n', hd, hd)*weight
        squared = float(Integrate(density_sample*diagnostic_ds, mesh))
        if not np.isfinite(squared) or squared < 0:
            raise RuntimeError("Invalid metric defect integral")
        return float(np.sqrt(squared))

    def record_metrics(step, time, field):
        jh = sample_tensor(grad(field).Trace())
        jt = np.array([1-time/2, 1-time/2, 1-2*time/3])[None, :, None]*j0_values
        je = jh-jt
        cross = jt.transpose(0, 2, 1) @ je
        exact_defect = cross+cross.transpose(0, 2, 1)+je.transpose(0, 2, 1) @ je
        exact_error = metric_norm(exact_defect, h_exact_values, mu_exact_values)
        target_metric.vec.data = (initial_discrete_metric.vec + time*ptg_const.vec
                                  + (0.5*time*time)*ptg_slope.vec)
        discrete_defect = jh.transpose(0, 2, 1) @ jh - sample_tensor(target_metric)
        discrete_error = metric_norm(discrete_defect, h_discrete_values, mu_discrete_values)
        metric_history.append((step, time, exact_error, discrete_error))

    for step, field in enumerate((Pos_2, Pos_1, Pos0)):
        record_metrics(step, step*tau, field)

    for n in range(step_n + 1, N_steps + 1):
        told = T if n == N_steps else n * tau

        Exa_ptg.vec.data = ptg_const.vec + told * ptg_slope.vec

        Pos_hat.vec.data = BaseVector(
            3 * Pos0.vec.FV().NumPy()
            - 3 * Pos_1.vec.FV().NumPy()
            + Pos_2.vec.FV().NumPy()
        )

        lhs.Assemble()
        rhs.Assemble()
        Solution.vec.data = lhs.mat.Inverse(inverse=inverse) * rhs.vec

        Pos1.vec.data = BaseVector(
            6 / 11 * (
                3 * Pos0.vec.FV().NumPy()
                - 3 / 2 * Pos_1.vec.FV().NumPy()
                + 1 / 3 * Pos_2.vec.FV().NumPy()
                + tau * Solution.components[0].vec.FV().NumPy()
            )
        )
        Pos_2.vec[:] = Pos_1.vec[:]
        Pos_1.vec[:] = Pos0.vec[:]
        Pos0.vec[:] = Pos1.vec[:]
        step_n = n
        record_metrics(n, told, Pos1)
        print(f"step {n}/{N_steps}, t={told:.6f}")


## 4. Errors

Graph errors and exact metric defects use the fixed exact reference-surface metric. Discrete defects compare with the FE-derived Regge target using the fixed Regge background. Both histories include every time node. Repeating the same parameters replaces the CSV in `results/`.


In [ ]:
quad_order_ref = 20

if step_n != N_steps or not np.isclose(told, T, rtol=1e-12, atol=0.0):
    raise RuntimeError("Run the time-stepping cell to T first")

mesh.UnsetDeformation()

axes2 = np.array([Ax, Ay, Az], dtype=float)**2
cp_normal = points / axes2
lam_cp = (
    np.sum((coords_3d - points) * cp_normal, axis=1)
    / np.sum(cp_normal**2, axis=1)
)
diag_B = 1.0 / (1.0 + lam_cp[:, None] / axes2)
Bn = diag_B * cp_normal
Da_values = (
    diag_B[:, :, None] * np.eye(3)
    - Bn[:, :, None] * Bn[:, None, :]
    / np.sum(cp_normal * Bn, axis=1)[:, None, None]
)

Da_entries = []
for i in range(3):
    for j in range(3):
        entry = GridFunction(fesir)
        entry.vec.FV().NumPy()[:] = Da_values[:, i, j]
        Da_entries.append(entry)
Da_cf = CF(tuple(Da_entries), dims=(3, 3))

nh = specialcf.normal(3)
Nh = OuterProduct(nh, nh)
Ph = Id(3) - Nh
J0_exact = Da_cf * Ph
Dt = CF((
    1-T/2, 0, 0,
    0, 1-T/2, 0,
    0, 0, 1-2*T/3,
), dims=(3, 3))
Jr_exact = Dt * J0_exact
Je = grad(Pos1).Trace() - Jr_exact
S = 0.5 * (Jr_exact.trans * Je + Je.trans * Jr_exact)
r_minus_l = (1-T)*X_ref0 + T*X_ref1
e = Pos1 - r_minus_l

G0_exact = J0_exact.trans * J0_exact
H_exact = Ph * Inv(G0_exact + Nh) * Ph
mu_exact = sqrt(Det(G0_exact + Nh))
ds_err = ds(intrules=irs)

L2_sq = Integrate(
    InnerProduct(e, e) * mu_exact * ds_err, mesh
)
strain_sq = Integrate(
    Trace(H_exact*S*H_exact*S) * mu_exact * ds_err, mesh
)
err_graph = sqrt(L2_sq + strain_sq)

g_exact = Jr_exact.trans * Jr_exact
metric_defect = 2*S + Je.trans * Je

Je_initial = grad(Xh0).Trace() - J0_exact
S_initial = 0.5 * (
    J0_exact.trans * Je_initial + Je_initial.trans * J0_exact
)
initial_metric_defect = 2*S_initial + Je_initial.trans * Je_initial

initial_metric_sq = Integrate(
    Trace(H_exact*initial_metric_defect*H_exact*initial_metric_defect)
    * mu_exact * ds_err,
    mesh,
)
metric_sq = Integrate(
    Trace(H_exact*metric_defect*H_exact*metric_defect) * mu_exact * ds_err,
    mesh,
)

err_initial_metric = sqrt(initial_metric_sq)
err_metric = sqrt(metric_sq)

history = np.asarray(metric_history)
if len(history) != N_steps+1 or not np.allclose(
        history[[0, -1], 2], [err_initial_metric, err_metric], rtol=1e-7, atol=1e-13):
    raise RuntimeError("Metric history is incomplete or disagrees with endpoint errors")
print(f"Total DOFs: {fesMix.ndof}; graph error: {err_graph:.12e}")
print(f"Exact metric defect: initial={err_initial_metric:.12e}, final={err_metric:.12e}")
for column, name in ((2, "Exact"), (3, "Discrete")):
    peak = int(np.argmax(history[:, column]))
    print(f"{name} metric defect: initial={history[0, column]:.12e}, "
          f"final={history[-1, column]:.12e}, max={history[peak, column]:.12e} "
          f"at t={history[peak, 1]:.6f}")
from pathlib import Path
result_dir = Path('results') / f'example_6_1_h{h:g}_r{r}_k{k}_tau{tau:g}_T{T:g}'
result_dir.mkdir(parents=True, exist_ok=True)
np.savetxt(result_dir/'metric_history.csv', history, delimiter=',',
           header='step,time,exact_metricerr,discrete_metricerr', comments='')


## 5. Optional surface view

Set `show_surface=True` to enable the Jupyter web widget.


In [ ]:
if show_surface:
    from ngsolve.webgui import Draw
    mesh.UnsetDeformation()
    deform = GridFunction(fesV)
    deform.Set(Pos1 - CF((x, y, z)), definedon=surface_region)
    mesh.SetDeformation(deform)
    Draw(mesh)
    mesh.UnsetDeformation()
